# Bronze Layer — Step 5: Data Profiling

Run this notebook **after** the bronze SDP pipeline has completed at least one run.
All profiling uses **PySpark** (not pandas) — these are Spark DataFrames.

**Answers the 6 required profiling questions:**
1. Is `claim_id` unique across all 1,000 records?
2. What % of claims have a missing `procedure_code`?
3. What % of claims have a missing `billed_amount`?
4. Are all `provider_id` values in claims present in providers?
5. What is the maximum `billed_amount`? Does it appear anomalous?
6. How many providers have a missing `location`?

**Widgets:**
- `catalog` — Unity Catalog catalog name (default: `healthcare`)
- `schema` — Schema name (default: `bronze`)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

dbutils.widgets.text("catalog", "healthcare", "Catalog")
dbutils.widgets.text("schema", "bronze", "Schema")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()

# Initialize all summary variables with sentinel so the summary cell
# is always safe to run even if an earlier cell failed or was skipped.
q1_result = "NOT RUN"
q2_result = "NOT RUN"
q3_result = "NOT RUN"
q4_result = "NOT RUN"
q5_result = "NOT RUN"
q6_result = "NOT RUN"

print(f"Profiling namespace: {CATALOG}.{SCHEMA}")

## Load bronze tables

In [ ]:
try:
    claims    = spark.table(f"{CATALOG}.{SCHEMA}.claims")
    providers = spark.table(f"{CATALOG}.{SCHEMA}.providers")
    diagnosis = spark.table(f"{CATALOG}.{SCHEMA}.diagnosis")
    cost      = spark.table(f"{CATALOG}.{SCHEMA}.cost")
except Exception as e:
    raise RuntimeError(
        f"Could not load bronze tables from {CATALOG}.{SCHEMA}. "
        "Run the bronze SDP pipeline first.\n" + str(e)
    )

counts = {
    "claims":    claims.count(),
    "providers": providers.count(),
    "diagnosis": diagnosis.count(),
    "cost":      cost.count(),
}
for name, n in counts.items():
    print(f"  {name}: {n} rows")

## 1. Row count check

In [ ]:
if counts["claims"] != 1000:
    raise AssertionError(
        f"Claims table has {counts['claims']} rows — expected 1,000. "
        "Check pipeline ingestion."
    )
print(f"✓ Claims: {counts['claims']} rows ingested (matches expected 1,000).")

## 2. Null analysis per column (all tables)

In [ ]:
def null_profile(df, table_name: str) -> list:
    total = df.count()
    rows = []
    for col_name in df.columns:
        if col_name.startswith("_"):
            continue  # Skip audit columns
        null_count = df.filter(
            F.col(col_name).isNull() | (F.trim(F.col(col_name).cast("string")) == "")
        ).count()
        rows.append({
            "table": table_name,
            "column": col_name,
            "null_count": null_count,
            "null_pct": round(100.0 * null_count / total, 2) if total > 0 else 0.0,
        })
    return rows

all_null_rows = []
for name, df in [("claims", claims), ("providers", providers), ("diagnosis", diagnosis), ("cost", cost)]:
    all_null_rows.extend(null_profile(df, name))

display(
    spark.createDataFrame(all_null_rows)
    .orderBy("table", F.col("null_pct").desc())
)

## Q1 — Is `claim_id` unique across all 1,000 records?

In [ ]:
total_claims = claims.count()
distinct_claims = claims.select("claim_id").distinct().count()
duplicate_count = total_claims - distinct_claims

print(f"Total rows        : {total_claims}")
print(f"Distinct claim_ids: {distinct_claims}")
print(f"Duplicates        : {duplicate_count}")

if duplicate_count == 0:
    q1_result = "PASS — claim_id is unique"
    print("\nRESULT: ✓ PASS — claim_id is unique across all records.")
else:
    q1_result = f"FAIL — {duplicate_count} duplicate claim_ids"
    dupes = (
        claims.groupBy("claim_id").count()
        .filter(F.col("count") > 1)
        .orderBy(F.col("count").desc())
        .limit(10)
    )
    print("\nRESULT: ✗ FAIL — Duplicate claim_ids found:")
    display(dupes)
    raise AssertionError(
        f"claim_id is NOT unique: {duplicate_count} duplicate rows detected. "
        "Investigate source CSV before proceeding."
    )

## Q2 & Q3 — Missing `procedure_code` and `billed_amount`

In [ ]:
total = claims.count()

missing_procedure = claims.filter(
    F.col("procedure_code").isNull() | (F.trim(F.col("procedure_code").cast("string")) == "")
).count()
missing_procedure_pct = round(100.0 * missing_procedure / total, 2)
q2_result = f"{missing_procedure_pct}% ({missing_procedure} rows) — {'⚠ auto-denial risk' if missing_procedure > 0 else '✓ OK'}"

missing_billed = claims.filter(F.col("billed_amount").isNull()).count()
missing_billed_pct = round(100.0 * missing_billed / total, 2)
q3_result = f"{missing_billed_pct}% ({missing_billed} rows) — {'⚠ unprocessable' if missing_billed > 0 else '✓ OK'}"

print(f"Q2 — Missing procedure_code: {missing_procedure} / {total} ({missing_procedure_pct}%)")
print(f"     {'⚠ WARNING — causes automatic claim denial' if missing_procedure > 0 else '✓ OK'}")

print(f"\nQ3 — Missing billed_amount: {missing_billed} / {total} ({missing_billed_pct}%)")
print(f"     {'⚠ WARNING — makes claim unprocessable' if missing_billed > 0 else '✓ OK'}")

## Q5 — `billed_amount` summary statistics

In [ ]:
amount_col = F.col("billed_amount").cast(DoubleType())

stats = claims.select(
    F.min(amount_col).alias("min_billed"),
    F.max(amount_col).alias("max_billed"),
    F.avg(amount_col).alias("avg_billed"),
    F.expr("percentile_approx(CAST(billed_amount AS DOUBLE), 0.95)").alias("p95_billed"),
    F.count(amount_col).alias("non_null_count"),
).collect()[0]

max_val = stats["max_billed"]
avg_val = stats["avg_billed"]

print("Q5 — billed_amount summary statistics:")
print(f"  Min     : ${stats['min_billed']:,.2f}" if stats["min_billed"] is not None else "  Min     : N/A (all null)")
print(f"  Max     : ${max_val:,.2f}" if max_val is not None else "  Max     : N/A")
print(f"  Average : ${avg_val:,.2f}" if avg_val is not None else "  Average : N/A")
print(f"  p95     : ${stats['p95_billed']:,.2f}" if stats["p95_billed"] is not None else "  p95     : N/A")
print(f"  Non-null: {stats['non_null_count']} / {total}")

anomaly_flag = ""
if max_val is not None and avg_val is not None and avg_val > 0:
    ratio = max_val / avg_val
    anomaly_flag = f" — ⚠ ANOMALY ({ratio:.1f}× avg)" if ratio > 10 else f" — ✓ normal ({ratio:.1f}× avg)"
    print(f"\n  {'⚠ ANOMALY' if ratio > 10 else '✓'}: max is {ratio:.1f}× the average.")

q5_result = (f"max=${max_val:,.2f}" if max_val is not None else "max=N/A") + anomaly_flag

## Q4 — Referential integrity: `claims.provider_id` → `providers.provider_id`

In [ ]:
provider_ids_in_providers = providers.select("provider_id").distinct()
orphaned = (
    claims.select("provider_id").distinct()
    .join(provider_ids_in_providers, on="provider_id", how="left_anti")
)
orphan_count = orphaned.count()

print("Q4 — Referential integrity (claims.provider_id → providers.provider_id):")
print(f"  Distinct provider_ids in claims   : {claims.select('provider_id').distinct().count()}")
print(f"  Distinct provider_ids in providers: {provider_ids_in_providers.count()}")
print(f"  Orphaned provider_ids in claims   : {orphan_count}")

if orphan_count == 0:
    q4_result = "PASS — all provider_ids resolve"
    print("\nRESULT: ✓ PASS — All provider_id values in claims exist in providers.")
else:
    q4_result = f"FAIL — {orphan_count} orphaned provider_ids"
    print("\nRESULT: ✗ FAIL — Orphaned provider_ids found:")
    display(orphaned.limit(20))
    raise AssertionError(
        f"{orphan_count} claim(s) reference provider_ids not in the providers table. "
        "Downstream silver/gold joins will produce NULLs or missing rows."
    )

## Q6 — Providers with missing `location`

In [ ]:
total_providers = providers.count()
missing_location_df = providers.filter(
    F.col("location").isNull() | (F.trim(F.col("location").cast("string")) == "")
)
missing_location = missing_location_df.count()
missing_location_pct = round(100.0 * missing_location / total_providers, 2)
q6_result = f"{missing_location} providers ({missing_location_pct}%) — {'⚠ admin rejection risk' if missing_location > 0 else '✓ OK'}"

print(f"Q6 — Providers with missing location: {missing_location} / {total_providers} ({missing_location_pct}%)")
print(f"     {'⚠ WARNING — missing location causes administrative rejection' if missing_location > 0 else '✓ OK'}")

if missing_location > 0:
    display(missing_location_df.limit(10))

## Overbilling detection — `billed_amount` vs regional benchmark

In [ ]:
claims_with_cost = (
    claims
    .filter(F.col("procedure_code").isNotNull())
    .join(cost, on="procedure_code", how="left")
    .withColumn(
        "overbilling_ratio",
        F.when(
            F.col("expected_cost").isNotNull() & (F.col("expected_cost").cast(DoubleType()) > 0),
            F.col("billed_amount").cast(DoubleType()) / F.col("expected_cost").cast(DoubleType()),
        ).otherwise(F.lit(None).cast(DoubleType()))
    )
)

overbilled = claims_with_cost.filter(F.col("overbilling_ratio") > 2.0)
overbilled_count = overbilled.count()

print(f"Overbilling check (billed_amount > 2× expected_cost):")
print(f"  Claims with procedure_code : {claims_with_cost.count()}")
print(f"  Claims overbilled (>2×)    : {overbilled_count}")

if overbilled_count > 0:
    print("  ⚠ Potential overbilling detected. Top examples:")
    display(
        overbilled
        .select("claim_id", "procedure_code", "billed_amount", "expected_cost", "region", "overbilling_ratio")
        .orderBy(F.col("overbilling_ratio").desc())
        .limit(10)
    )
else:
    print("  ✓ No overbilling detected relative to cost benchmarks.")

## Profiling Summary

This cell is safe to run even if earlier cells were skipped — all variables are initialized at the top.

In [ ]:
summary = [
    {"#": "Q1", "check": "claim_id unique?",           "result": q1_result},
    {"#": "Q2", "check": "missing procedure_code %",   "result": q2_result},
    {"#": "Q3", "check": "missing billed_amount %",    "result": q3_result},
    {"#": "Q4", "check": "provider_id ref integrity",  "result": q4_result},
    {"#": "Q5", "check": "max billed_amount",          "result": q5_result},
    {"#": "Q6", "check": "providers missing location", "result": q6_result},
]

display(spark.createDataFrame(summary))

not_run = [r["#"] for r in summary if r["result"] == "NOT RUN"]
if not_run:
    print(f"\n⚠ The following checks did not run: {not_run}. Re-run the notebook top-to-bottom.")
else:
    print("\n✓ All 6 profiling checks completed. Review warnings before proceeding to silver layer.")